# 08 - Race simulator

This is where the pieces come together: the lap-time model, the safety car model, and the costs a lap-time model leaves out (pit stops, safety car laps, the first lap). The simulator runs one driver's race many times and gives a spread of finishing times. Everything here is fit on 2022 and 2023 and tested on 2024. I haven't touched 2025.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from f1lab.features import drying_races, wet_races
from f1lab.safety_car import lap_status
from f1lab.simulate import Stint, build_simulator, candidate_strategies, describe, rank_strategies
from f1lab.backtest import constant_pace_version, driver_races, rank_agreement, replay, score

raw = pd.read_parquet("../data/laps.parquet")
clean = pd.read_parquet("../data/laps_clean.parquet")
sim = build_simulator(raw, clean, seasons=(2022, 2023))   # fitted on 2022-2023 only
plans = driver_races(raw, clean, seasons=(2024,))          # real 2024 races to replay
print(f"{len(plans)} driver-races from {len({(p.season, p.round) for p in plans})} races")

227 driver-races from 19 races


## What a lap-time model leaves out

In [2]:
pl = sim.pit_loss
medians = pd.Series(pl.track_median_)
print(f"pit stop on green flags: median {pl.overall_:.1f} s, track medians run from {medians.min():.1f} to {medians.max():.1f} s")
print(f"pit stop under a safety car or VSC: median {np.median(pl.neutral_pool_):.1f} s, middle 80% from "
      f"{np.quantile(pl.neutral_pool_, .1):.1f} to {np.quantile(pl.neutral_pool_, .9):.1f} s")
print(f"lap 1 takes {sim.pools['lap1'].mean():.3f} times a normal lap on average")

pit stop on green flags: median 22.6 s, track medians run from 19.7 to 27.1 s
pit stop under a safety car or VSC: median 19.4 s, middle 80% from 5.7 to 44.1 s
lap 1 takes 1.143 times a normal lap on average


A green-flag pit stop costs about 22.6 s, anywhere from 19.7 to 27.1 s depending on the track. Under a safety car it's cheaper on average (19.4 s) but the spread is huge, from 5.7 to 44.1 s, because it depends on when the field slows down relative to the stop. Lap 1 takes about 14% longer than a normal lap.

In [3]:
# safety car laps: how slow, by season
skip = drying_races(clean) | wet_races(raw)
dry = raw[[(s, r) not in skip for s, r in zip(raw["season"], raw["round"])]]
pace = clean.groupby(["season", "round", "driver"])["lap_time_s"].median().rename("pace")
d = dry.join(pace, on=["season", "round", "driver"])
d = d[d["pace"].notna() & d["lap_time_s"].notna() & ~d["is_pit_in"] & ~d["is_pit_out"]]
d = d.assign(ratio=d["lap_time_s"] / d["pace"]).merge(lap_status(dry)[["season", "round", "lap_number", "rank"]], on=["season", "round", "lap_number"])
d[d["rank"] == 2].groupby("season")["ratio"].agg(laps="size", mean="mean").round(3)

,laps,mean
season,,
2022,993,1.328
2023,594,1.493
2024,527,1.453
2025,716,1.445


Safety car laps run at about 1.45 to 1.49 times a normal lap in 2023, 2024 and 2025, but only 1.33 in 2022. That jump is the data source changing, not the sport. The 2022 laps come from FastF1, which marks a lap as safety car if any part of it was, so it includes partial laps. Because of that, the simulator learns its safety car costs from the OpenF1 seasons only.

## Replaying 2024

In [4]:
full = replay(sim, plans, n=400)
flat = replay(constant_pace_version(sim), plans, n=400)
naive = pd.Series([p.actual_total - p.base_pace * p.race_laps for p in plans])
table = pd.DataFrame({"simulator": score(full), "constant pace": score(flat)})
table.loc["rank agreement within races"] = [rank_agreement(full), rank_agreement(flat)]
print(f"just base pace x laps, no costs: median abs error {naive.abs().median():.1f} s, mean error {naive.mean():.1f} s")
table.round(3)

just base pace x laps, no costs: median abs error 77.0 s, mean error 153.6 s


,simulator,constant pace
median abs error (s),18.081,17.886
mean error (s),-10.897,-12.263
RMSE (s),47.273,48.742
inside 80% interval,0.802,0.802
rank agreement within races,0.883,0.897


I replay each 2024 race with the strategy every driver actually ran and the safety car laps that actually happened, then compare the predicted total race time with the real one. Guessing base pace times laps misses by a median of 77 s, so the costs matter. With them the simulator is off by a median of 18 s on a race of about 5,500 s, and its 80% interval contains the real time 80.2% of the time.

The surprise is that the tyre and fuel model doesn't improve the total. A version with no tyre or fuel effect does about the same (17.9 s median error) and ranks drivers a bit better within a race (0.897 against 0.883). That makes sense in hindsight. A driver's base pace is their own median lap from that race, so it already soaks up the average shape, and the total barely depends on how time is spread across stints. Where the tyre model should matter is in comparing strategies, which this test is blunt about.

In [5]:
neutral = {(p.season, p.round): int((p.status > 0).sum()) for p in plans}
full["neutral laps"] = [neutral[(s, r)] for s, r in zip(full["season"], full["round"])]
by = full.groupby(pd.cut(full["neutral laps"], [-1, 0, 5, 10, 100], labels=["none", "1-5", "6-10", "over 10"]), observed=True)
by.agg(driver_races=("error", "size"), median_abs_error=("error", lambda s: s.abs().median()),
       mean_error=("error", "mean"), inside_80=("inside", "mean")).round(2)

,driver_races,median_abs_error,mean_error,inside_80
neutral laps,,,,
none,103,10.67,-5.28,0.90
1-5,67,17.15,5.81,0.85
6-10,43,98.02,-63.60,0.42
over 10,14,26.38,29.64,1.00


Most of the error comes from safety cars. In races with no safety car or VSC lap the median error is 10.7 s and 90% of results land inside the 80% interval, so it's a little conservative there. Races with 6 to 10 neutralised laps are much worse (98 s median, only 42% inside the interval). Safety car pace varies a lot from race to race, from 1.30 to 1.60 times a normal lap in 2024, and a single average can't follow that. Those errors are shared by every driver in a race, so they mostly cancel when you compare strategies within one race.

In [6]:
# are the intervals honest? coverage at several nominal levels
cover = {f"{lvl:.0%}": replay(sim, plans, n=400, level=lvl)["inside"].mean() for lvl in (0.5, 0.8, 0.9)}
pd.Series(cover, name="share of races inside the interval").round(3)

50%    0.529
80%    0.802
90%    0.894
Name: share of races inside the interval, dtype: float64

Over all races the intervals are honest: the 50% interval contains 52.9% of results, the 80% one 80.2% and the 90% one 89.4%.

## Does the simulator rank strategies fairly?

In [7]:
# races with no safety car / VSC lap: error by number of stops, with a bootstrap range on the mean
quiet = full[full["neutral laps"] == 0]
rng = np.random.default_rng(0)
rows = {}
for stops, g in quiet.groupby("stops"):
    boots = [rng.choice(g["error"].to_numpy(), len(g)).mean() for _ in range(2000)]
    rows[f"{stops} stop"] = {"driver-races": len(g), "mean error (s)": g["error"].mean(),
                             "low": np.quantile(boots, .05), "high": np.quantile(boots, .95)}
pd.DataFrame(rows).T.round(1)

,driver-races,mean error (s),low,high
1 stop,34.0,-2.9,-6.8,1.2
2 stop,66.0,-6.4,-9.4,-3.3
3 stop,3.0,-6.5,-25.2,12.1


In races with no safety car I split the error by number of stops. Actual times come out 2.9 s faster than predicted for one-stoppers and 6.4 s faster for two-stoppers, on a race of about 5,500 s. So the simulator makes a two-stop race look about 3.5 s slower, relative to a one-stop race, than it really was. That's small, and the ranges overlap a bit. There are only 3 three-stop races here, so I can't say anything about those.

## Searching for a strategy

In [8]:
# Sakhir 2024: what would the simulator have picked for the race winner?
winner = min((p for p in plans if p.round == 1), key=lambda p: p.actual_total)
others = sorted((p for p in plans if p.round == 1), key=lambda p: p.actual_total)[:5]
print("race winner:", winner.driver, describe(winner.strategy), f"| base pace {winner.base_pace:.1f} s | {winner.race_laps} laps")
candidates = candidate_strategies(winner.race_laps, max_stops=3, min_stint=10, step=6)
actual = [p.strategy for p in others]
ranked = rank_strategies(sim, winner.track, winner.race_laps, winner.base_pace, winner.track_temp, winner.air_temp,
                         plans=candidates + actual, n=200, react=True)
print(len(candidates), "candidate plans searched")
ranked.head(8).round(2)

race winner: VER S17 > H20 > S20 | base pace 95.5 s | 57 laps


1182 candidate plans searched


,plan,stops,mean_s,sd_s,behind_best_s,p_beats_best
0,H34 > S23,1,5615.18,125.33,0.00,0.00
1,M34 > S23,1,5615.40,125.33,0.22,0.00
2,H40 > S17,1,5615.61,124.76,0.43,0.40
3,S40 > M17,1,5615.73,124.84,0.56,0.40
4,M40 > S17,1,5615.83,124.76,0.65,0.38
5,S34 > M23,1,5615.91,125.32,0.74,0.00
6,S40 > H17,1,5616.04,124.81,0.86,0.38
7,S34 > H23,1,5616.22,125.30,1.04,0.00


I asked the simulator for the best plan for the 2024 Sakhir winner. Verstappen ran a two-stop, S17 > H20 > S20. Out of 1,182 candidate plans the simulator prefers one-stops, with H34 > S23 best at about 5,615 s on average. The first few plans are within a second of each other, so the order inside that group means nothing.

In [9]:
is_actual = ranked["plan"].isin([describe(s) for s in actual])
ranked[is_actual].round(2)

,plan,stops,mean_s,sd_s,behind_best_s,p_beats_best
69,S12 > H24 > S21,2,5628.99,135.17,13.81,0.40
98,S17 > H20 > S20,2,5629.59,135.20,14.41,0.38
216,S11 > H23 > H23,2,5630.89,134.17,15.71,0.36
245,S14 > H21 > H22,2,5631.17,134.62,15.99,0.37
311,S11 > H20 > H26,2,5632.55,134.20,17.37,0.35


What actually ran sits about 14 s behind the best plan on average. The plans the top five finishers ran rank between 69th and 311th out of 1,187.

In [10]:
# when does the winner's two-stop beat the simulator's favourite one-stop?
pool = {describe(s): s for s in candidates + actual}
best_plan, winner_plan = pool[ranked["plan"].iloc[0]], winner.strategy
kw = dict(track=winner.track, race_laps=winner.race_laps, base_pace=winner.base_pace,
          track_temp=winner.track_temp, air_temp=winner.air_temp, n=4000, seed=7, react=True)
a = sim.simulate(strategy=best_plan, **kw)
b = sim.simulate(strategy=winner_plan, **kw)
wins = b.totals < a.totals
had_sc = a.neutralisations > 0
print(f"runs with a safety car or VSC: {had_sc.mean():.0%}")
print(f"two-stop beats the one-stop in {wins.mean():.0%} of all runs")
print(f"  ...in runs with a safety car:    {wins[had_sc].mean():.0%}")
print(f"  ...in runs with no safety car:   {wins[~had_sc].mean():.0%}")
print(f"average gap, no safety car runs: two-stop is {np.mean(b.totals[~had_sc] - a.totals[~had_sc]):.1f} s slower")

runs with a safety car or VSC: 74%
two-stop beats the one-stop in 35% of all runs
  ...in runs with a safety car:    38%
  ...in runs with no safety car:   27%
average gap, no safety car runs: two-stop is 16.8 s slower


That's the average, though. Run to run the total moves by about 125 s (standard deviation), and the winner's two-stop beats the simulator's favourite one-stop in 35% of runs. It's 38% when there's a safety car and still 27% when there isn't, so a safety car helps the two-stop a bit but most of the flips are ordinary random variation in stint pace and pit stop time.

I'd treat this as a demonstration, not advice. The replay checks the simulator against strategies teams really ran. A one-stop at Sakhir with a long soft stint at the end is outside that, and a wear model that's nearly linear can't see a tyre cliff.